# IMDb Movie Review Sentiment Analysis using Simple RNN

This clean portfolio notebook documents the production project workflow:

1. Load a GitHub-safe sample.
2. Inspect sentiment and review-length distributions.
3. Apply the same text cleaning and chunked sequence logic used by the app.
4. Load the saved Keras Simple RNN and tokenizer.
5. Score sample reviews.
6. Review held-out metrics, baselines, diagnostics, and error analysis.

The full raw IMDb dataset is loaded programmatically only during retraining.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import clean_review_frame
from src.sentiment_pipeline import load_artifacts, predict_reviews


## Load the included sample


In [ ]:
sample = pd.read_csv(PROJECT_ROOT / "data" / "sample_reviews.csv")
sample.head()


## Validate text data


In [ ]:
cleaned_sample, quality_report = clean_review_frame(
    sample,
    text_column="review",
)
quality_report


## Load the saved model and tokenizer


In [ ]:
artifacts = load_artifacts()
artifacts.model_metadata


## Generate sentiment predictions


In [ ]:
sample_predictions, sequence_reports = predict_reviews(
    cleaned_sample["review"].tolist(),
    artifacts,
)
sample_predictions[
    [
        "review",
        "predicted_sentiment",
        "positive_probability",
        "confidence",
        "confidence_band",
    ]
].head(10)


## Held-out performance


In [ ]:
metrics = json.loads(
    (PROJECT_ROOT / "outputs" / "model_metrics.json").read_text()
)
comparison = pd.read_csv(
    PROJECT_ROOT / "outputs" / "model_comparison.csv"
)
metrics, comparison


## Confusion matrix and model diagnostics


In [ ]:
from IPython.display import Image, display

for file_name in [
    "confusion_matrix.png",
    "roc_curve.png",
    "precision_recall_curve.png",
    "training_accuracy.png",
    "training_loss.png",
    "baseline_comparison.png",
]:
    display(Image(filename=str(PROJECT_ROOT / "outputs" / file_name)))


## Error analysis


In [ ]:
errors = pd.read_csv(PROJECT_ROOT / "outputs" / "error_analysis.csv")
errors[[
    "error_type",
    "positive_probability",
    "text_length",
    "review_excerpt",
]].head(15)


## Interpretation

The Simple RNN improves materially over the original two-epoch implementation,
but the TF-IDF logistic-regression baseline remains stronger on the limited
training subset. This is an important analytical result rather than something
to hide: architecture complexity does not guarantee better generalization.

Likely error causes include sarcasm, mixed sentiment, long-distance context,
rare language, and the limited memory of a basic Simple RNN.
